## Graph visualization

---

### Create pandas dataframe from graphviz svg output

#### Imports

In [3]:
import os
import math
from xml.etree import ElementTree as ET
import urllib.parse
from urllib.parse import urlsplit
from urllib.request import pathname2url
import json
import codecs
import subprocess
import networkx as nx
from networkx.algorithms import bipartite
import pandas as pd
import numpy as np
from numpy import dot
from numpy.linalg import norm
from scipy.stats import entropy
from collections import Counter
import locale
import requests
from bs4 import BeautifulSoup
import uuid
from time import *
locale.setlocale(locale.LC_ALL, 'de-DE.utf-8')
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

---

### Convert svg to pandas dataframe

#### Structure of the pandas dataframe

In [ ]:
#columns for nodes:
        #   id               node id
        #   tf               term frequency
        #   df               document frequency
        #   
        #   cx_fd            x coordinate for node center, fd stands for force-directed layout
        #   cy_fd            y coordinate for node center, fd stands for force-directed layout
        #   rx_fd            x radius, fd stands for force-directed layout
        #   ry_fd            y radius, fd stands for force-directed layout

#columns for edges:
        #   id               edge id
        #   start_id         id of start node
        #   end_id           id of end node
        #   weight           weight
        #
        #   start_x_fd       x coordinate for start point, fd stands for force-directed layout
        #   start_y_fd       y coordinate for start point, fd stands for force-directed layout
        #   end_x_fd         x coordinate for end point, fd stands for force-directed layout
        #   end_y_fd         y coordinate for end point, fd stands for force-directed layout
        #   start_x_node     x coordinate for start at node center
        #   start_y_node     y coordinate for start at node center
        #   end_x_node       x coordinate for end at node center
        #   end_y_node       y coordinate for end at node center

#### Read the svg file

In [ ]:
# set path to svg file
svg_file = 'graphs/first_graph.svg'
# read graph svg file
with open(svg_file, 'r', encoding='utf-8') as s:
    ss = ET.parse(s)
svg_in = ss.getroot()
# define namespaces
ns = {'svg': 'http://www.w3.org/2000/svg'}
# read transform and viewbox attributes
transform = svg_in.find('svg:g[@id="graph0"]',ns).get('transform')
viewbox   = svg_in.get('viewBox')
print (f'transform: {transform}')
print (f'viewbox: {viewbox}')

transform: scale(1 1) rotate(0) translate(4 2979.9)
viewbox: 0.00 0.00 3035.63 2983.90


#### Get svg node coordinates

In [64]:
node_coords = {node.find("svg:title", ns).text: ( node.find("svg:ellipse", ns).attrib['cx'], \
                                                node.find("svg:ellipse", ns).attrib['cy'], \
                                                node.find("svg:ellipse", ns).attrib['rx'], \
                                                node.find("svg:ellipse", ns).attrib['ry']  \
                                                )  for node in svg_in.findall(".//svg:g[@class='node']", ns)}
text_coords = {node.find("svg:title", ns).text: (node.find("svg:text", ns).attrib['x'], \
                                                node.find("svg:text", ns).attrib['y']  \
                                                ) for node in svg_in.findall(".//svg:g[@class='node']", ns)}

#### Create pandas dataframe for graph nodes

In [65]:
node_ids = list(node_coords.keys())
nds = pd.DataFrame({  \
                'cx_fd': [node_coords.get(k)[0] for k in node_ids], \
                'cy_fd': [node_coords.get(k)[1] for k in node_ids], \
                'rx_fd': [node_coords.get(k)[2] for k in node_ids], \
                'ry_fd': [node_coords.get(k)[3] for k in node_ids], \
                'txt_x': [text_coords.get(k)[0] for k in node_ids], \
                'txt_y': [text_coords.get(k)[1] for k in node_ids],  \
                'label': [k for k in node_ids], \
                }, \
                index = node_ids \
                )

#### Get svg path coordinates

In [66]:
path_ids = [id.attrib['id'] for id in svg_in.findall(".//svg:g[@class='edge']", ns)]
path_coords = [p.attrib['d'] for p in svg_in.findall(".//svg:path", ns)]

#### Create pandas dataframe for graph edges

In [69]:
eds = pd.DataFrame({
            'start_id'    : [u.split('--')[0] for u in path_ids], \
            'end_id'      : [v.split('--')[1] for v in path_ids], \
            'start_x_fd'  : [coords.split(',')[0][1:]           for coords in path_coords], \
            'start_y_fd'  : [coords.split(',')[1].split('C')[0] for coords in path_coords], \
            'end_x_fd'    : [coords.split(' ')[2].split(',')[0] for coords in path_coords], \
            'end_y_fd'    : [coords.split(' ')[2].split(',')[1] for coords in path_coords], \
            'start_x_node': [node_coords[p.split('--')[0]][0]   for p in path_ids], \
            'start_y_node': [node_coords[p.split('--')[0]][1]   for p in path_ids], \
            'end_x_node'  : [node_coords[p.split('--')[1]][0]   for p in path_ids], \
            'end_y_node'  : [node_coords[p.split('--')[1]][1]   for p in path_ids] \
        }, \
        index = path_ids) 
print (eds.head())

                   start_id      end_id start_x_fd start_y_fd end_x_fd  end_y_fd start_x_node start_y_node end_x_node end_y_node
arrive--execution    arrive   execution    2616.85   -2313.13  2269.79  -1965.75      2647.18     -2343.48    2239.26    -1935.2
arrive--punishment   arrive  punishment    2605.94   -2330.79  2020.81  -2150.66      2647.18     -2343.48    1979.84   -2138.05
arrive--incur        arrive       incur    2614.58   -2315.56  2380.47  -2115.02      2647.18     -2343.48    2347.91   -2087.14
arrive--citizens     arrive    citizens    2613.83   -2370.77  2319.09  -2611.93      2647.18     -2343.48    2285.83   -2639.15
arrive--official     arrive    official    2604.93   -2335.73  2344.95  -2288.03      2647.18     -2343.48    2302.62   -2280.27


### Create svg file from pandas dataframe

In [ ]:
# compute initial font size
fontsize = math.ceil(float(max(nds['rx_fd']))/10)
print ('font size: ', fontsize)
# create svg root element
svg_out_attr = {'xmlns':'http://www.w3.org/2000/svg', 'xmlns:xlink':'http://www.w3.org/1999/xlink', 'version':'1.1', 'viewBox':viewbox}
svg_out = ET.Element('svg', attrib=svg_out_attr)
# create svg graph
g0_node_attr = {'id':'graph0', 'transform':transform}
g0_node = ET.SubElement(svg_out,'g',attrib=g0_node_attr)    
# graph edges
for u,v,att in graph.edges(data=True):
    ed_id       = att.get('id')
    edge_attr   = {'class':'edge', 'id':ed_id, 'style':'cursor: pointer;'}
    edge        = ET.SubElement(rcp_g, 'g', attrib=edge_attr)
    start_x     = eds.at[ed_id,'start_x_node']
    start_y     = eds.at[ed_id,'start_y_node']
    end_x       = eds.at[ed_id,'end_x_node']
    end_y       = eds.at[ed_id,'end_y_node']
    pt_coor     = f"M{start_x},{start_y}L{end_x},{end_y}"
    xx = wgt_growing.get(ed_id)
    if xx == 1:
        path_attr   = {'fill':'none', 'stroke': 'black', 'd':pt_coor}
    elif xx == 2: 
        path_attr   = {'fill':'none', 'stroke': 'black', 'stroke-width':'2', 'd':pt_coor}
    elif xx > 2: 
        path_attr   = {'fill':'none', 'stroke': 'red', 'stroke-width':'2', 'd':pt_coor}
    else: 
        path_attr   = {'fill':'none', 'stroke': 'black', 'd':pt_coor}
    path        = ET.SubElement(edge,'path',path_attr)
# graph nodes
font_size = fontsize * scale
for n in graph.nodes():
    node_attr   = {'class':'node', 'id':n, 'data-sub':self.nds.at[n,'sub'], 'style':'cursor: pointer;'}
    node        = ET.SubElement(rcp_g, 'g', attrib=node_attr)
    title       = ET.SubElement(node, 'title')
    title.text  = f"#occ: {occ_growing.get(n)}"
    ellip_class = f"i-{self.nds.at[n,'class']}" 
    x           = 36*(1 + 3*math.sqrt(occ_growing.get(n)))
    rx          = str(round(x*2, 0)/2)
    ry          = rx              
    ellip_attr  = {'class':ellip_class, 'cx':self.nds.at[n,'cx_fd'], 'cy':self.nds.at[n,'cy_fd'], 'rx':rx, 'ry':ry}
    ET.SubElement(node, 'ellipse', attrib=ellip_attr)
    text_attr   = {'x':self.nds.at[n,'txt_x'], 'y':self.nds.at[n,'txt_y'], 'style':f"text-anchor: middle; font-family: Arial Narrow; font-size: {font_size}px;"}
    text        = ET.SubElement(node, 'text', attrib=text_attr)
    text.text   = self.nds.at[n,'name'] 
# return svg
return svg_out